# Touchline Lab: исследование АПЛ 2015/16
Исследуем связь наблюдаемых игровых показателей с исходом матча. Причинность не предполагается. Перед запуском выполните загрузку, ingest и обучение согласно README. Ноутбук читает ту же SQLite-базу и артефакты, что и приложение.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "touchline").exists():
    root = root.parent
if not (root / "touchline").exists():
    raise RuntimeError("Open the notebook from the project root or notebooks directory")
sys.path.insert(0, str(root))

In [2]:
from touchline.data import factor_frame, query
from touchline.modeling import FACTORS, SHOT_FEATURES
from touchline.research import audit

pd.set_option("display.max_columns", 12)

## 1. Качество и полнота
Проверяем диапазоны xG, координаты и соответствие голевых событий счёту. Автоголы учитываются один раз через Own Goal Against. Выход координат одного удара за границу поля сохранён и задокументирован; этот удар исключён из собственного xG.

In [3]:
quality = audit()
print(json.dumps(quality, indent=2))
frame = factor_frame()
print("Teams:", frame.team.nunique())
print("Matches per team:", frame.groupby("team").size().unique().tolist())
assert frame.groupby("match_id").size().eq(2).all()

{
  "matches": 380,
  "events": 1313773,
  "shots": 9908,
  "score_mismatches": 0,
  "invalid_shot_xg": 0,
  "invalid_shot_coordinates": 1,
  "gross_coordinate_errors": 0,
  "own_goals": 38,
  "excluded_own_xg_shots": 1
}
Teams: 20
Matches per team: [38]


## 2. Сезонные различия: SQL и pandas
Голы − xG — описательный остаток; он не равен чистой удаче или мастерству. Сравнение xG не определяет, кто заслуживал победы.

In [4]:
league = query("""SELECT team, SUM(points) AS points, SUM(goals) AS goals,
       ROUND(SUM(xg), 2) AS xg, ROUND(SUM(goals-xg),2) AS residual
       FROM team_match GROUP BY team ORDER BY points DESC, SUM(goals-conceded) DESC""")
print(league.to_string(index=False))
print("\nСредние показатели по исходам:")
print(
    frame.groupby("result")[["xg_diff", "shots_diff", "box_entries_diff", "pressures_diff"]].mean().round(3)
)

                team  points  goals    xg  residual
      Leicester City      81     68 66.28      1.72
             Arsenal      71     65 65.09     -0.09
   Tottenham Hotspur      70     69 62.36      6.64
     Manchester City      66     71 64.56      6.44
   Manchester United      66     49 42.20      6.80
         Southampton      63     59 54.22      4.78
     West Ham United      62     65 54.53     10.47
           Liverpool      60     63 58.37      4.63
          Stoke City      51     41 40.37      0.63
             Chelsea      50     59 53.58      5.42
             Everton      47     59 50.61      8.39
        Swansea City      47     42 42.15     -0.15
             Watford      45     40 44.08     -4.08
West Bromwich Albion      43     34 38.63     -4.63
      Crystal Palace      42     39 42.38     -3.38
     AFC Bournemouth      42     45 40.35      4.65
          Sunderland      39     48 42.49      5.51
    Newcastle United      37     44 40.67      3.33
        Norw

![Сезонные результаты](../reports/figures/season.png)
## 3. Оконная функция SQL: только предыдущие матчи
Текущий матч исключён рамкой `ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING`. Этот запрос иллюстрирует построение исторического признака и не используется послематчевой моделью.

In [5]:
rolling = query("""SELECT date, team, opponent, points,
    AVG(points) OVER (PARTITION BY team_id ORDER BY date, match_id
                     ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING) AS prior_five_ppg
    FROM team_match ORDER BY team, date""")
print(rolling[rolling.team.eq("Leicester City")].head(12).to_string(index=False))

      date           team             opponent  points  prior_five_ppg
2015-08-08 Leicester City           Sunderland       3             NaN
2015-08-15 Leicester City      West Ham United       3        3.000000
2015-08-22 Leicester City    Tottenham Hotspur       1        3.000000
2015-08-29 Leicester City      AFC Bournemouth       1        2.333333
2015-09-13 Leicester City          Aston Villa       3        2.000000
2015-09-19 Leicester City           Stoke City       1        2.200000
2015-09-26 Leicester City              Arsenal       0        1.800000
2015-10-03 Leicester City         Norwich City       3        1.200000
2015-10-17 Leicester City          Southampton       1        1.600000
2015-10-24 Leicester City       Crystal Palace       3        1.600000
2015-10-31 Leicester City West Bromwich Albion       3        1.600000
2015-11-07 Leicester City              Watford       3        2.000000


## 4. Проверка ML-эксперимента
Разделение по уникальным датам. Выбор модели выполняется по validation log loss. Test не используется для выбора. События одного матча не разделяются между выборками.

In [6]:
metrics = json.loads((root / "artifacts/metrics.json").read_text(encoding="utf-8"))
predictions = pd.read_csv(root / "artifacts/shot_predictions.csv")
print("Cutoffs:", metrics["cutoffs"])
print("Match splits:", metrics["split_matches"])
print("Selected xG model:", metrics["xg"]["selected"])
rows = [{"model": name, **value["test"]} for name, value in metrics["xg"]["candidates"].items()]
rows.append({"model": "constant_rate", **metrics["xg"]["baseline"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))
assert predictions.groupby("match_id")["split"].nunique().max() == 1
assert not {"goal", "statsbomb_xg", "end_x", "end_y"}.intersection(SHOT_FEATURES)
assert not {"goals", "conceded", "result"}.intersection(FACTORS)

Cutoffs: {'train_end': '2016-01-24', 'validation_end': '2016-04-09'}
Match splits: {'train': 230, 'validation': 90, 'test': 60}
Selected xG model: logistic
            model  log_loss  brier  accuracy  roc_auc
         logistic    0.2894 0.0836    0.8957   0.7814
gradient_boosting    0.2900 0.0831    0.8975   0.7759
    constant_rate    0.3454 0.0974    0.8908   0.5000


![Проверка моделей](../reports/figures/models.png)
## Интерпретация
Собственный xG сравнивается с постоянной вероятностью гола и внешней оценкой StatsBomb. Модель исхода использует итоговую статистику и поэтому не является предматчевым прогнозом. Состояние счёта и уровень соперника ограничивают причинные интерпретации. Полные результаты, интервалы и ограничения: [исследование](../reports/research.md).